In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path


def load_participant_info(bilgi_path):
    """Extract participant information from Bilgi.txt"""
    if not os.path.exists(bilgi_path):
        return {}
        
    info = {}
    with open(bilgi_path, 'r', encoding='utf-8') as f:
        for line in f:
            if ':' in line:
                key, value = line.strip().split(':', 1)
                # Map Turkish keys to English
                key_mapping = {
                    'Ad': 'name',
                    'Soyad': 'surname',
                    'Cinsiyet': 'gender',
                    'Kilo': 'weight',
                    'Boy': 'height',
                    'Yaş': 'age',
                    'Tel': 'telephone'
                }
                info[key_mapping.get(key, key)] = value.strip()
    
    return info



def get_dataset_summary(data_collection):
    """
    Get a summary of the fall detection dataset
    
    Args:
        data_collection: List of dictionaries containing the loaded data
        
    Returns:
        dict: Summary statistics of the dataset
    """
    summary = {
        'total_samples': len(data_collection),
        'unique_participants': set(),
        'gender_distribution': {},
        'activity_types': {},
        'fall_activities': {},
        'daily_activities': {},
        'sensors_per_sample': {},
        'total_sensors_available': set()
    }
    
    # Analyze each sample
    for sample in data_collection:
        # Participant information
        summary['unique_participants'].add(sample['participant_id'])
        
        # Gender distribution (if available)
        if 'gender' in sample['participant_info']:
            gender = sample['participant_info']['gender']
            summary['gender_distribution'][gender] = summary['gender_distribution'].get(gender, 0) + 1
        
        # Activity classification
        activity_code = sample['activity_code']
        activity_name = sample['activity_name']
        summary['activity_types'][activity_name] = summary['activity_types'].get(activity_name, 0) + 1
        
        # Classify as fall or daily activity
        if activity_code.startswith('9'):
            summary['fall_activities'][activity_name] = summary['fall_activities'].get(activity_name, 0) + 1
        else:
            summary['daily_activities'][activity_name] = summary['daily_activities'].get(activity_name, 0) + 1
        
        # Sensor information
        available_sensors = list(sample['sensor_data'].keys())
        sensor_key = tuple(sorted(available_sensors))
        summary['sensors_per_sample'][sensor_key] = summary['sensors_per_sample'].get(sensor_key, 0) + 1
        
        # Track all available sensors
        for sensor in available_sensors:
            summary['total_sensors_available'].add(sensor)
    
    # Convert sets to lists for better display
    summary['unique_participants'] = sorted(list(summary['unique_participants']))
    summary['total_sensors_available'] = sorted(list(summary['total_sensors_available']))
    
    # Calculate percentages
    total_samples = summary['total_samples']
    summary['fall_activity_percentage'] = (sum(summary['fall_activities'].values()) / total_samples) * 100
    summary['daily_activity_percentage'] = (sum(summary['daily_activities'].values()) / total_samples) * 100
    
    return summary




def fix_sensor_loading(base_dir="./Tests"):
    """
    Fix sensor loading to handle all sensor files correctly
    
    Returns:
        list: Updated data collection with fixed sensor data
    """
    data_collection = []
    
    # Complete sensor mapping including the problematic ones
    sensor_locations = {
        '340506': 'head',
        '340527': 'chest', 
        '340535': 'waist',
        '340537': 'right_wrist',
        '340539': 'right_thigh',
        '340540': 'right_ankle',
        '535': 'waist',  # It appears this is also waist sensor (shortened ID)
        '506': 'head',   # Possibly shortened IDs
        '527': 'chest',
        '537': 'right_wrist',
        '539': 'right_thigh',
        '540': 'right_ankle'
    }
    
    for participant_folder in os.listdir(base_dir):
        if not os.path.isdir(os.path.join(base_dir, participant_folder)):
            continue
            
        participant_id = participant_folder
        if not participant_id.isdigit():
            continue
            
        participant_path = os.path.join(base_dir, participant_folder)
        bilgi_path = os.path.join(participant_path, "Bilgi.txt")
        participant_info = load_participant_info(bilgi_path)
        
        testler_export_path = os.path.join(participant_path, "Testler Export")
        if not os.path.exists(testler_export_path):
            continue
            
        for activity_folder in os.listdir(testler_export_path):
            if not os.path.isdir(os.path.join(testler_export_path, activity_folder)):
                continue
                
            activity_code = activity_folder.split('-')[0]
            activity_path = os.path.join(testler_export_path, activity_folder)
            
            for test_folder in os.listdir(activity_path):
                test_path = os.path.join(activity_path, test_folder)
                
                if not os.path.isdir(test_path):
                    continue
                
                # Fix sensor loading
                sensor_data = {}
                for sensor_file in os.listdir(test_path):
                    if sensor_file.endswith('.txt'):
                        sensor_id = sensor_file.replace('.txt', '')
                        sensor_path = os.path.join(test_path, sensor_file)
                        
                        # Try to read the file with different approaches
                        try:
                            # First, try to find header
                            header_line = None
                            with open(sensor_path, 'r') as f:
                                for line_num, line in enumerate(f):
                                    if line.startswith('Counter'):
                                        header_line = line_num
                                        break
                            
                            if header_line is not None:
                                data = pd.read_csv(sensor_path, delimiter='\t', skiprows=header_line)
                            else:
                                # If no header, check if it has 24 columns (including counter)
                                data = pd.read_csv(sensor_path, delimiter='\t')
                                if len(data.columns) == 24:
                                    # Manually set column names
                                    columns = ['Counter', 'Temperature', 'VelInc_X', 'VelInc_Y', 'VelInc_Z', 
                                              'OriInc_w', 'OriInc_x', 'OriInc_y', 'OriInc_z', 
                                              'Acc_X', 'Acc_Y', 'Acc_Z', 
                                              'Gyr_X', 'Gyr_Y', 'Gyr_Z', 
                                              'Mag_X', 'Mag_Y', 'Mag_Z', 
                                              'Pressure', 'Roll', 'Pitch', 'Yaw', 'RSSI']
                                    data.columns = columns
                            
                            # Map sensor ID to location
                            location = sensor_locations.get(sensor_id, 'unknown')
                            sensor_data[location] = data
                            
                        except Exception as e:
                            print(f"Error reading {sensor_path}: {e}")
                
                record = {
                    'participant_id': participant_id,
                    'participant_info': participant_info,
                    'activity_code': activity_code,
                    'activity_name': activity_folder,
                    'test_id': test_folder,
                    'sensor_data': sensor_data
                }
                
                data_collection.append(record)
    
    return data_collection

# Run this to get corrected data
print("Reloading data with fixed sensor mapping...")
fixed_data = fix_sensor_loading()
print(f"Loaded {len(fixed_data)} samples")

# Check if we still have unknown sensors
unknown_count = 0
for sample in fixed_data:
    if 'unknown' in sample['sensor_data']:
        unknown_count += 1

print(f"Unknown sensors remaining: {unknown_count}")

# Get updated summary
if unknown_count == 0:
    summary = get_dataset_summary(fixed_data)
    print("\nUpdated sensor summary:")
    print(f"Available sensors: {summary['total_sensors_available']}")

Reloading data with fixed sensor mapping...
Error reading ./Tests\103\Testler Export\911-GeriyeDogruSagli\Test_5\535.txt: Length mismatch: Expected axis has 24 elements, new values have 23 elements
Loaded 3326 samples
Unknown sensors remaining: 1


In [2]:
import numpy as np
import pandas as pd
from scipy.fft import fft
from scipy import stats
import os

def extract_robust_features(data_collection, sensor_selection='waist_only'):
    """
    Extract features from sensor data for fall detection while preventing data leakage
    
    Args:
        data_collection: List of dictionaries containing the loaded sensor data
        sensor_selection: Strategy for sensor selection ('waist_only', 'waist_chest', or 'all')
        
    Returns:
        tuple: (features DataFrame, labels Series)
    """
    import numpy as np
    from scipy.fft import fft
    from scipy import stats
    
    # Define sensors based on selection strategy
    if sensor_selection == 'waist_only':
        focus_sensors = ['waist']
    elif sensor_selection == 'waist_chest':
        focus_sensors = ['waist', 'chest']
    else:  # 'all'
        focus_sensors = ['waist', 'chest', 'head', 'right_wrist', 'right_thigh', 'right_ankle']
    
    print(f"Using sensor selection: {sensor_selection} ({', '.join(focus_sensors)})")
    
    # Prepare dataframe
    features = []
    labels = []
    
    # Track which samples are processed successfully
    processed_count = 0
    skipped_count = 0
    
    # Create a unique ID for each sample to maintain data integrity in train/test splits
    sample_id = 0
    
    for sample in data_collection:
        try:
            # Create base feature dict - include only essential metadata
            # IMPORTANT: Do not include activity_code or activity_name to prevent data leakage
            sample_id += 1
            feature_dict = {
                'sample_id': sample_id,
                'participant_id': sample['participant_id'],
            }
            
            # Create binary label (1 for falls, 0 for non-falls)
            is_fall = 1 if sample['activity_code'].startswith('9') else 0
            feature_dict['is_fall'] = is_fall
            
            # Extract activity label for evaluation - store separately from features
            activity_label = int(sample['activity_code'])
            
            # Extract features from each sensor
            all_sensors_present = True
            
            for sensor in focus_sensors:
                if sensor not in sample['sensor_data']:
                    all_sensors_present = False
                    continue
                
                # Get sensor data
                sensor_df = sample['sensor_data'][sensor]
                
                if sensor_df.empty:
                    all_sensors_present = False
                    continue
                
                # Calculate features for this sensor
                try:
                    # 1. Time-domain statistical features
                    # Accelerometer features
                    for axis in ['X', 'Y', 'Z']:
                        acc_col = f'Acc_{axis}'
                        if acc_col not in sensor_df.columns:
                            continue
                            
                        values = sensor_df[acc_col].values
                        
                        # Basic statistics
                        feature_dict[f'{sensor}_acc_{axis}_mean'] = np.mean(values)
                        feature_dict[f'{sensor}_acc_{axis}_std'] = np.std(values)
                        feature_dict[f'{sensor}_acc_{axis}_max'] = np.max(values)
                        feature_dict[f'{sensor}_acc_{axis}_min'] = np.min(values)
                        feature_dict[f'{sensor}_acc_{axis}_range'] = np.max(values) - np.min(values)
                        
                        # Percentiles
                        feature_dict[f'{sensor}_acc_{axis}_25th'] = np.percentile(values, 25)
                        feature_dict[f'{sensor}_acc_{axis}_75th'] = np.percentile(values, 75)
                        
                        # Zero crossing rate
                        zero_crossings = np.where(np.diff(np.signbit(values)))[0]
                        feature_dict[f'{sensor}_acc_{axis}_zero_crossings'] = len(zero_crossings)
                    
                    # 2. Overall acceleration magnitude (vector magnitude)
                    if all(col in sensor_df.columns for col in ['Acc_X', 'Acc_Y', 'Acc_Z']):
                        acc_mag = np.sqrt(
                            sensor_df['Acc_X']**2 + 
                            sensor_df['Acc_Y']**2 + 
                            sensor_df['Acc_Z']**2
                        )
                        
                        feature_dict[f'{sensor}_acc_mag_mean'] = np.mean(acc_mag)
                        feature_dict[f'{sensor}_acc_mag_std'] = np.std(acc_mag)
                        feature_dict[f'{sensor}_acc_mag_max'] = np.max(acc_mag)
                        feature_dict[f'{sensor}_acc_mag_min'] = np.min(acc_mag)
                        feature_dict[f'{sensor}_acc_mag_range'] = np.max(acc_mag) - np.min(acc_mag)
                        
                        # 3. Signal Magnitude Area (SMA)
                        window_size = min(50, len(sensor_df))  # Use smaller window if data is short
                        sma = 0
                        for i in range(len(sensor_df) - window_size + 1):
                            window = acc_mag[i:i+window_size]
                            sma += np.sum(window) / window_size
                        
                        # Normalize by number of windows
                        if len(sensor_df) > window_size:
                            sma /= (len(sensor_df) - window_size + 1)
                        
                        feature_dict[f'{sensor}_sma'] = sma
                        
                        # 4. Jerk (derivative of acceleration)
                        jerk_x = np.diff(sensor_df['Acc_X'])
                        jerk_y = np.diff(sensor_df['Acc_Y'])
                        jerk_z = np.diff(sensor_df['Acc_Z'])
                        
                        # Jerk magnitude
                        jerk_mag = np.sqrt(jerk_x**2 + jerk_y**2 + jerk_z**2)
                        
                        feature_dict[f'{sensor}_jerk_mag_mean'] = np.mean(jerk_mag)
                        feature_dict[f'{sensor}_jerk_mag_max'] = np.max(jerk_mag)
                        feature_dict[f'{sensor}_jerk_mag_std'] = np.std(jerk_mag)
                    
                    # 5. Frequency-domain features
                    if len(sensor_df) >= 64:  # Need sufficient data for FFT
                        for axis in ['X', 'Y', 'Z']:
                            acc_col = f'Acc_{axis}'
                            if acc_col not in sensor_df.columns:
                                continue
                                
                            # Apply FFT
                            fft_values = fft(sensor_df[acc_col].values[:64])
                            # Take the magnitude of the first 10 frequencies
                            fft_mag = np.abs(fft_values[:10])
                            
                            # Store dominant frequency
                            feature_dict[f'{sensor}_acc_{axis}_dom_freq'] = np.argmax(fft_mag[1:]) + 1
                            # Energy in frequency domain
                            feature_dict[f'{sensor}_acc_{axis}_freq_energy'] = np.sum(fft_mag**2)
                    
                    # 6. Gyroscope features (angular velocity)
                    if all(col in sensor_df.columns for col in ['Gyr_X', 'Gyr_Y', 'Gyr_Z']):
                        gyr_mag = np.sqrt(
                            sensor_df['Gyr_X']**2 + 
                            sensor_df['Gyr_Y']**2 + 
                            sensor_df['Gyr_Z']**2
                        )
                        
                        feature_dict[f'{sensor}_gyr_mag_mean'] = np.mean(gyr_mag)
                        feature_dict[f'{sensor}_gyr_mag_std'] = np.std(gyr_mag)
                        feature_dict[f'{sensor}_gyr_mag_max'] = np.max(gyr_mag)
                        
                        # Angular velocity peak-to-peak
                        feature_dict[f'{sensor}_gyr_mag_p2p'] = np.max(gyr_mag) - np.min(gyr_mag)
                    
                    # 7. Orientation features if available
                    for angle in ['Roll', 'Pitch', 'Yaw']:
                        if angle in sensor_df.columns:
                            values = sensor_df[angle].values
                            feature_dict[f'{sensor}_{angle.lower()}_mean'] = np.mean(values)
                            feature_dict[f'{sensor}_{angle.lower()}_range'] = np.max(values) - np.min(values)
                            feature_dict[f'{sensor}_{angle.lower()}_std'] = np.std(values)
                    
                    # 8. Correlation between axes
                    if all(col in sensor_df.columns for col in ['Acc_X', 'Acc_Y', 'Acc_Z']):
                        corr_xy = np.corrcoef(sensor_df['Acc_X'], sensor_df['Acc_Y'])[0, 1]
                        corr_xz = np.corrcoef(sensor_df['Acc_X'], sensor_df['Acc_Z'])[0, 1]
                        corr_yz = np.corrcoef(sensor_df['Acc_Y'], sensor_df['Acc_Z'])[0, 1]
                        
                        feature_dict[f'{sensor}_acc_corr_xy'] = corr_xy
                        feature_dict[f'{sensor}_acc_corr_xz'] = corr_xz
                        feature_dict[f'{sensor}_acc_corr_yz'] = corr_yz
                
                except Exception as e:
                    print(f"Error processing sensor {sensor} for sample {sample['participant_id']}-{sample['activity_code']}: {e}")
                    all_sensors_present = False
            
            # Only add samples where all required sensors were successfully processed
            if all_sensors_present:
                features.append(feature_dict)
                labels.append(activity_label)
                processed_count += 1
            else:
                skipped_count += 1
                
        except Exception as e:
            print(f"Error processing sample {sample['participant_id']}-{sample['activity_code']}: {e}")
            skipped_count += 1
    
    print(f"Successfully processed {processed_count} samples")
    print(f"Skipped {skipped_count} samples")
    
    # Convert to pandas DataFrame
    features_df = pd.DataFrame(features)
    labels_series = pd.Series(labels, name='activity_label')
    
    # Print feature set statistics
    print(f"Feature set shape: {features_df.shape}")
    print(f"Number of features: {features_df.shape[1]}")
    print(f"Number of fall samples: {features_df['is_fall'].sum()}")
    print(f"Number of non-fall samples: {len(features_df) - features_df['is_fall'].sum()}")
    
    return features_df, labels_series

In [3]:
def robust_train_test_split(features_df, labels_series, test_size=0.2, random_state=42):
    """
    Split data ensuring that samples from the same participant don't appear in both train and test sets
    
    Args:
        features_df: DataFrame containing features
        labels_series: Series containing activity labels
        test_size: Proportion of participants to use for testing
        random_state: Random seed for reproducibility
        
    Returns:
        tuple: (X_train, X_test, y_train, y_test)
    """
    # Get unique participants
    participants = features_df['participant_id'].unique()
    
    # Shuffle participants
    np.random.seed(random_state)
    np.random.shuffle(participants)
    
    # Split participants into train and test groups
    n_test = int(len(participants) * test_size)
    test_participants = participants[:n_test]
    train_participants = participants[n_test:]
    
    print(f"Training participants: {len(train_participants)}")
    print(f"Testing participants: {len(test_participants)}")
    
    # Create masks for train and test sets
    train_mask = features_df['participant_id'].isin(train_participants)
    test_mask = features_df['participant_id'].isin(test_participants)
    
    # Get feature columns (excluding metadata)
    metadata_cols = ['sample_id', 'participant_id', 'is_fall']
    feature_cols = [col for col in features_df.columns if col not in metadata_cols]
    
    # Split data
    X_train = features_df.loc[train_mask, feature_cols]
    X_test = features_df.loc[test_mask, feature_cols]
    y_train = features_df.loc[train_mask, 'is_fall']
    y_test = features_df.loc[test_mask, 'is_fall']
    
    print(f"Training samples: {len(X_train)}")
    print(f"Testing samples: {len(X_test)}")
    print(f"Fall samples in training: {y_train.sum()}")
    print(f"Non-fall samples in training: {len(y_train) - y_train.sum()}")
    print(f"Fall samples in testing: {y_test.sum()}")
    print(f"Non-fall samples in testing: {len(y_test) - y_test.sum()}")
    
    return X_train, X_test, y_train, y_test

In [4]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def participant_cross_validation(features_df, model, n_splits=5, random_state=42):
    """
    Perform cross-validation at the participant level
    
    Args:
        features_df: DataFrame containing features
        model: Model to evaluate
        n_splits: Number of cross-validation folds
        random_state: Random seed for reproducibility
        
    Returns:
        list: Cross-validation scores
    """
    # Get unique participants
    participants = features_df['participant_id'].unique()
    
    # Create KFold object
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    # Get feature columns (excluding metadata)
    metadata_cols = ['sample_id', 'participant_id', 'is_fall']
    feature_cols = [col for col in features_df.columns if col not in metadata_cols]
    
    cv_scores = []
    fold = 1
    
    # Perform cross-validation
    for train_idx, test_idx in kf.split(participants):
        # Get train and test participants
        train_participants = participants[train_idx]
        test_participants = participants[test_idx]
        
        # Create masks for train and test sets
        train_mask = features_df['participant_id'].isin(train_participants)
        test_mask = features_df['participant_id'].isin(test_participants)
        
        # Split data
        X_train = features_df.loc[train_mask, feature_cols]
        X_test = features_df.loc[test_mask, feature_cols]
        y_train = features_df.loc[train_mask, 'is_fall']
        y_test = features_df.loc[test_mask, 'is_fall']
        
        # Train model
        model.fit(X_train, y_train)
        
        # Predict
        y_pred = model.predict(X_test)
        
        # Calculate accuracy
        accuracy = accuracy_score(y_test, y_pred)
        cv_scores.append(accuracy)
        
        print(f"Fold {fold} - Accuracy: {accuracy:.4f}")
        print(f"Fold {fold} - Classification Report:")
        print(classification_report(y_test, y_pred))
        
        fold += 1
    
    print(f"Cross-validation scores: {cv_scores}")
    print(f"Mean CV accuracy: {np.mean(cv_scores):.4f}")
    
    return cv_scores

In [5]:
def analyze_fall_detection_results(X_train, X_test, y_train, y_test, model, feature_names):
    """
    Analyze feature importance and visualize class separation
    
    Args:
        X_train, X_test, y_train, y_test: Training and testing data
        model: Trained model
        feature_names: List of feature names
        
    Returns:
        DataFrame: Feature importance scores
    """
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    # Train model if not already trained
    if not hasattr(model, "feature_importances_"):
        model.fit(X_train, y_train)
    
    # Get feature importances if available
    if hasattr(model, "feature_importances_"):
        # Get feature importances
        importances = model.feature_importances_
        
        # Create DataFrame for feature importances
        feature_importance = pd.DataFrame({
            'Feature': feature_names,
            'Importance': importances
        })
        
        # Sort by importance
        feature_importance = feature_importance.sort_values('Importance', ascending=False)
        
        # Plot top 20 features
        plt.figure(figsize=(12, 8))
        sns.barplot(x='Importance', y='Feature', data=feature_importance.head(20))
        plt.title('Top 20 Most Important Features')
        plt.tight_layout()
        plt.show()
    
    # PCA visualization
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_test)
    
    plt.figure(figsize=(10, 8))
    plt.scatter(X_pca[y_test==0, 0], X_pca[y_test==0, 1], alpha=0.5, label='Non-Fall')
    plt.scatter(X_pca[y_test==1, 0], X_pca[y_test==1, 1], alpha=0.5, label='Fall')
    plt.legend()
    plt.title('PCA Visualization of Fall vs. Non-Fall Data')
    plt.show()
    
    # t-SNE visualization for better separation
    tsne = TSNE(n_components=2, random_state=42)
    X_tsne = tsne.fit_transform(X_test)
    
    plt.figure(figsize=(10, 8))
    plt.scatter(X_tsne[y_test==0, 0], X_tsne[y_test==0, 1], alpha=0.5, label='Non-Fall')
    plt.scatter(X_tsne[y_test==1, 0], X_tsne[y_test==1, 1], alpha=0.5, label='Fall')
    plt.legend()
    plt.title('t-SNE Visualization of Fall vs. Non-Fall Data')
    plt.show()
    
    # Confusion matrix
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
               xticklabels=['Non-Fall', 'Fall'], 
               yticklabels=['Non-Fall', 'Fall'])
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()
    
    # Classification report
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    
    return feature_importance if hasattr(model, "feature_importances_") else None

In [7]:
from sklearn.ensemble import RandomForestClassifier

# 1. Extract features
features_df, labels_series = extract_robust_features(fixed_data, sensor_selection='waist_only')

# 2. Split data by participant
X_train, X_test, y_train, y_test = robust_train_test_split(features_df, labels_series, test_size=0.2)

# 3. Train a Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 4. Evaluate the model
y_pred = rf_model.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# 5. Perform cross-validation
cv_scores = participant_cross_validation(features_df, RandomForestClassifier(n_estimators=100, random_state=42))



Using sensor selection: waist_only (waist)
Successfully processed 3324 samples
Skipped 2 samples
Feature set shape: (3324, 58)
Number of features: 58
Number of fall samples: 1842
Number of non-fall samples: 1482
Training participants: 14
Testing participants: 3
Training samples: 2763
Testing samples: 561
Fall samples in training: 1540
Non-fall samples in training: 1223
Fall samples in testing: 302
Non-fall samples in testing: 259
Random Forest Accuracy: 0.9964349376114082
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       259
           1       1.00      0.99      1.00       302

    accuracy                           1.00       561
   macro avg       1.00      1.00      1.00       561
weighted avg       1.00      1.00      1.00       561

Fold 1 - Accuracy: 0.9974
Fold 1 - Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       355
           1       1.00      